# 32 · Publication figures V2 — render-only

**Purpose:** reproduce the six manuscript publication figures from already-frozen/post-freeze results. This notebook does **not** fit a model, recalibrate probabilities, retune thresholds, redesign v3, or open a new scientific experiment.

Evidence scope remains 72 hours only. Cycle-25 v3 evidence is diagnostic/post-hoc; supplementary 2026 is the clean future evaluation of frozen v3; Phase-H 2026 diagnostics are post-hoc.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ROOT = Path.cwd().resolve()
while ROOT.name != 'gray-box-solar-flare-forecasting' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == 'gray-box-solar-flare-forecasting', 'Run inside repository clone'
OUT = ROOT / 'outputs' / 'manuscript_publication_figures_v2'
OUT.mkdir(parents=True, exist_ok=True)

CROSS = ROOT/'results/72h_graybox/20261006/cross_cycle_reliability_audit/cross_cycle_branch_metrics.csv'
ROLL  = ROOT/'results/72h_graybox/20261006/rolling_operational_replay_v1/final_replay_summary.csv'
SEV   = ROOT/'results/72h_graybox/20261006/real_outage_structure_v1/graybox_exposure_by_episode_severity.csv'
BASE  = ROOT/'results/72h_graybox/20261006/baseline_suite_v2/baseline_metrics.csv'
DELTA = ROOT/'results/72h_graybox/20261006/baseline_suite_v2/baseline_paired_bootstrap_deltas_key.csv'

for p in [CROSS,ROLL,SEV,BASE,DELTA]:
    assert p.is_file(), p

plt.rcParams.update({'font.family':'serif','font.size':10,'axes.titlesize':11,'axes.labelsize':10})

def save(fig, stem):
    fig.savefig(OUT/f'{stem}.png', dpi=450, bbox_inches='tight')
    fig.savefig(OUT/f'{stem}.svg', bbox_inches='tight')
    plt.close(fig)


## Figure 1 — architecture and evidence chronology


In [ ]:
fig,ax=plt.subplots(figsize=(10.8,6.9)); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
def bx(x,y,w,h,text,fs=9,ls='-'):
    ax.add_patch(patches.FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.008',fill=False,linewidth=1.1,linestyle=ls))
    ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=fs)
def ar(x1,y1,x2,y2,ls='-'):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',lw=1.0,linestyle=ls))
ax.text(.5,.955,'72-hour Gray-Box trust architecture and evidence chronology',ha='center',va='top',fontsize=13,fontweight='bold')
bx(.04,.71,.18,.11,'Temporal SHARP\nGRU\n$p_S$'); bx(.04,.54,.18,.11,'Temporal AIA\nCNN–GRU\n$p_A$')
bx(.29,.625,.18,.11,'Fixed fusion\n$p_F=0.5p_S+0.5p_A$'); ar(.22,.765,.29,.68); ar(.22,.595,.29,.68)
bx(.53,.75,.19,.09,'Calibration +\nMondrian conformal'); bx(.53,.61,.19,.09,'Agreement signals\nspread / entropy / gap')
bx(.53,.47,.19,.09,'Physical applicability\nMPSV + MD²',ls='--'); ar(.47,.68,.53,.795); ar(.47,.68,.53,.655); ar(.47,.68,.53,.515,ls='--')
ax.text(.625,.438,'monitor only — not a v3 gate',ha='center',fontsize=8)
bx(.80,.72,.15,.085,'NORMAL\nissue fusion'); bx(.80,.59,.15,.085,'DEGRADED\nissue SHARP'); bx(.80,.46,.15,.085,'ABSTAIN\nwithhold')
ar(.72,.795,.80,.762); ar(.72,.655,.80,.632); bx(.785,.345,.18,.075,'v3 routing:\nNORMAL → DEGRADED → ABSTAIN',fs=8)
ax.text(.875,.318,'ABSTAIN only if SHARP unavailable',ha='center',fontsize=7.8)
ax.text(.5,.285,'Evidence chronology',ha='center',fontsize=11,fontweight='bold'); y=.215; ax.hlines(y,.06,.94,linewidth=1.1)
xs=np.array([.08,.22,.35,.48,.61,.76,.92]); ax.vlines(xs,y-.012,y+.012,linewidth=1.0)
labels=['Train\n2010–13','Model val.\nH1 2014','Prob. cal.\nH2 2014','Conformal cal.\nH1 2015','Policy val.\nH2 2015–2019','Cycle 25\n2021–25','2026\nfuture eval.']
for x,l in zip(xs,labels): ax.text(x,.18,l,ha='center',va='top',fontsize=8)
ax.text(.5,.055,'Cycle 25 informed fallback-first v3 semantics without new numerical trust-threshold tuning; supplementary 2026 evaluates frozen v3. Phase H is post-freeze/post-hoc.',ha='center',fontsize=8)
save(fig,'Figure1_architecture_chronology_v2')


## Figure 2 — cross-regime degradation


In [ ]:
cross=pd.read_csv(CROSS)
regmap={'cycle24_earlier_development':'Earlier Cycle 24','cycle25_2021_2025_diagnostic':'Cycle 25','supplementary_2026_locked':'2026'}
branchmap={'sharp':'SHARP','aia':'AIA','sharp_aia':'Fusion'}
r=cross[cross['grouping'].eq('regime')].copy(); r['regime']=r['group'].map(regmap); r['branch_name']=r['branch'].map(branchmap)
train_prev=0.0576877980145392
r['bss']=1-r['brier']/(train_prev*(1-train_prev))
metrics=[('tss','TSS','(a) Discrimination'),('bss','Brier skill score','(b) Probabilistic skill'),('flare_conformal_coverage','Flare-class coverage','(c) Conformal transfer')]
for k,(col,ylab,title) in enumerate(metrics,1):
    fig,ax=plt.subplots(figsize=(4.2,4.0)); x=np.arange(3); width=.24
    for j,b in enumerate(['SHARP','AIA','Fusion']):
        vals=[float(r[(r.regime==rg)&(r.branch_name==b)][col].iloc[0]) for rg in ['Earlier Cycle 24','Cycle 25','2026']]
        ax.bar(x+(j-1)*width,vals,width=width,label=b)
    ax.set_xticks(x,['Earlier\nCycle 24','Cycle 25','2026']); ax.set_ylabel(ylab); ax.set_title(title); ax.axhline(0,lw=.7); ax.legend(frameon=False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.grid(axis='y',alpha=.18)
    save(fig,f'Figure2{k}_panel_v2')


## Figure 3 — reliability paradox


In [ ]:
# Frozen H1c common-support medians; these are descriptive post-hoc diagnostics, not newly estimated here.
seed=np.array([0.012219,0.006433,0.001626]); entropy=np.array([0.138411,0.118344,0.048715]); gap=np.array([0.027830,0.019232,0.004042])
fusion_cov=np.array([0.8378378378,0.6922279793,0.3457661290]); x=np.arange(3)
fig,ax=plt.subplots(figsize=(5.4,4.2))
for name,v in [('Fusion seed spread',seed),('Fusion entropy',entropy),('SHARP–AIA gap',gap)]: ax.plot(x,v/v[0],marker='o',label=name)
ax.set_xticks(x,['Earlier Cycle 24','Cycle 25','2026']); ax.set_ylim(0,1.08); ax.set_ylabel('Relative to earlier Cycle-24 value'); ax.set_title('(a) Internal agreement becomes tighter'); ax.legend(frameon=False); ax.grid(axis='y',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
save(fig,'Figure3a_agreement_v2')
fig,ax=plt.subplots(figsize=(4.2,4.2)); ax.plot(x,fusion_cov,marker='o'); ax.set_xticks(x,['Earlier\nCycle 24','Cycle 25','2026']); ax.set_ylim(0,.9); ax.set_ylabel('Fusion flare-class conformal coverage'); ax.set_title('(b) Rare-flare reliability falls'); ax.grid(axis='y',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for xi,yi in zip(x,fusion_cov): ax.text(xi,yi+.035,f'{yi:.3f}',ha='center',fontsize=8)
save(fig,'Figure3b_coverage_v2')


## Figure 4 — structured outage and routing


In [ ]:
sev=pd.read_csv(SEV); roll=pd.read_csv(ROLL)
fig,ax=plt.subplots(figsize=(4.5,4.2)); x=np.arange(len(sev)); bars=ax.bar(x,sev['mapped_graybox_cases'])
ax.set_xticks(x,['Isolated','Short\n2–4','Medium\n5–10','Long\n>10']); ax.set_ylabel('Mapped Gray-Box cases'); ax.set_title('(a) AIA failures are clustered'); ax.set_yscale('log'); ax.grid(axis='y',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for rr,v in zip(bars,sev['mapped_graybox_cases']): ax.text(rr.get_x()+rr.get_width()/2,v*1.12,f'{int(v):,}',ha='center',fontsize=8)
save(fig,'Figure4a_outage_severity_v2')
sel=pd.concat([roll[(roll.scenario=='nominal_frozen_inputs')&(roll.role=='retrospective_cycle25')],roll[(roll.scenario=='real_aia_incident_replay')&(roll.role=='retrospective_cycle25')],roll[(roll.scenario=='nominal_frozen_inputs')&(roll.role=='supplementary_2026')]])
fig,ax=plt.subplots(figsize=(5.0,4.2)); x=np.arange(3); ax.bar(x,sel.normal_rate,label='NORMAL'); ax.bar(x,sel.degraded_rate,bottom=sel.normal_rate,label='DEGRADED')
ax.set_xticks(x,['Nominal\nCycle 25','AIA incident\nCycle 25','Nominal\n2026']); ax.set_ylim(0,1.03); ax.set_ylabel('Fraction of issued forecasts'); ax.set_title('(b) Missing AIA routes to SHARP fallback',pad=10); ax.legend(frameon=False,ncol=2,loc='lower center',bbox_to_anchor=(.5,-.28)); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
save(fig,'Figure4b_routing_v2')


## Figure 5 — same-support baseline forest plot


In [ ]:
# Point metrics come from the archived executed baseline suite. Bootstrap interval endpoints are the frozen Notebook-31 results.
base=pd.read_csv(BASE)
roles=['policy_validation','retrospective_cycle25','supplementary_2026']; role_names=['Policy validation','Cycle 25','2026']
models=['logistic_latest_sharp','frozen_sharp','frozen_aia','frozen_fusion']; model_names=['Logistic latest SHARP','Frozen SHARP','Frozen AIA','Frozen fusion']
ci={
 'policy_validation':[(.360207,.771009),(.416103,.783625),(.196499,.425435),(.474674,.744185)],
 'retrospective_cycle25':[(.429055,.567900),(.475960,.602300),(.270159,.411194),(.414536,.557110)],
 'supplementary_2026':[(.082601,.445377),(.121814,.515235),(-.011606,.170915),(.096356,.478373)]}
fig,ax=plt.subplots(figsize=(7.2,6.3)); base_y=np.array([8,4,0],float); offsets=np.array([.75,.25,-.25,-.75]); markers=['o','s','^','D']
for j,(m,mn) in enumerate(zip(models,model_names)):
    xs=np.array([base[(base.role==r)&(base.model==m)].tss.iloc[0] for r in roles]); lo=np.array([ci[r][j][0] for r in roles]); hi=np.array([ci[r][j][1] for r in roles]); ys=base_y+offsets[j]
    ax.errorbar(xs,ys,xerr=np.vstack([xs-lo,hi-xs]),fmt=markers[j],capsize=3,label=mn,linestyle='none')
for y,n in zip(base_y,role_names): ax.text(-.045,y,n,ha='right',va='center',fontweight='bold')
ax.axvline(0,lw=.7); ax.set_xlim(-.02,.82); ax.set_ylim(-1.4,9.6); ax.set_yticks([]); ax.set_xlabel('TSS at validation-chosen/frozen threshold'); ax.set_title('Same-support baselines with active-region 95% intervals'); ax.grid(axis='x',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.spines['left'].set_visible(False)
ax.text(.44,3.0,'Cycle 25 paired deltas:\nSHARP − logistic  +0.039 [0.0003, 0.0819]\nfusion − SHARP  −0.053 [−0.091, −0.013]',fontsize=8,va='top'); ax.legend(frameon=False,ncol=2,loc='upper center',bbox_to_anchor=(.5,-.08))
save(fig,'Figure5_baseline_forest_v2')


## Figure 6 — physical applicability sensitivity


In [ ]:
# Frozen H1d/H1c summary values; no new Phase-H fit is performed.
md=np.array([[10.716,8.299,9.839],[12.673,9.608,11.600],[16.731,9.385,11.410]])
fig,ax=plt.subplots(figsize=(5.3,4.2)); x=np.arange(3)
for i,n in enumerate(['Earlier Cycle 24','Cycle 25','2026']): ax.plot(x,md[i],marker='o',label=n)
ax.set_xticks(x,['All H1b axes','No gradients\nfamily','No gradients\nlatest']); ax.set_ylabel('Median squared Mahalanobis distance'); ax.set_title('(a) 2026 elevation is gradient-dominated'); ax.legend(frameon=False); ax.grid(axis='y',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
save(fig,'Figure6a_gradient_sensitivity_v2')
nonflare=[11.187,13.072,17.235]; flare=[7.075,9.293,11.057]
fig,ax=plt.subplots(figsize=(4.7,4.2)); x=np.arange(3); w=.32; ax.bar(x-w/2,nonflare,w,label='Non-flare'); ax.bar(x+w/2,flare,w,label='Flare'); ax.set_xticks(x,['Earlier\nCycle 24','Cycle 25','2026']); ax.set_ylabel('Median squared Mahalanobis distance'); ax.set_title('(b) Distance is not an error score'); ax.legend(frameon=False); ax.grid(axis='y',alpha=.18); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
save(fig,'Figure6b_class_conditioned_v2')


## Claim boundary

This rendering notebook changes no scientific state. It adds no H2 gate, no new model, and no 24-hour or 3-hour result. The figure values remain subordinate to the frozen result folders and the claim-to-artifact map.
